# Qwen3-1.7B LoRA SFT — Kaggle Fine-Tuning Notebook

## How to run on Kaggle

### Attach private datasets
In Kaggle → Settings → Data, attach the following four datasets:

| Dataset | Suggested mount path | Contents |
|---|---|---|
| **sft-code** | `/kaggle/input/sft-code/` | The `sft/` directory (excluding `.venv*`, `work/`) |
| **qwen3-weights** | `/kaggle/input/qwen3-weights/` | `sft/model_snapshot/Qwen3-1.7B/` (full model weights) |
| **sft-data** | `/kaggle/input/sft-data/` | `dialogues_train.jsonl` and `dialogues_val.jsonl` |
| **behavior-set** *(optional)* | `/kaggle/input/behavior-set/` | `behaviors.jsonl` (5% mix; omit to run without) |

To resume after a session cut: publish `/kaggle/working/checkpoints` as a dataset, attach it, and set `RESUME_FROM` in the Config cell.

### GPU accelerator
Settings → Accelerator → **GPU T4 x2** (recommended) or **P100**.  
Only one GPU is used (`CUDA_VISIBLE_DEVICES=0`). The second T4 is ignored.

### Internet
Turn **Internet ON** only during the pip cell (cell 3). After that you may turn it off. All model files are loaded locally (`local_files_only=True`, `HF_HUB_OFFLINE=1`).

### Run sequence
Run all cells top to bottom. Outputs go to `/kaggle/working/`.  
The environment guard (cell 2) will halt immediately if not on Kaggle, no CUDA is found, or free VRAM < 6 GB.

### Resuming
1. After a session cut, go to Output → publish `/kaggle/working/checkpoints` as a new dataset.
2. Attach that dataset (e.g. at `/kaggle/input/checkpoints-resume/`).
3. Set `RESUME_FROM` in the Config cell to the checkpoint directory path.
4. Re-run all cells. Training will resume from that checkpoint.

### DRY_RUN
Set `DRY_RUN = True` in the Config cell to test the notebook without a GPU.  
In DRY_RUN mode: the environment guard is skipped, ~20 synthetic dialogues are generated instead of reading real JSONL files, and training/validation cells are skipped. Tests T1–T6 still run on the synthetic data.

### Pilot experiments
- **E0** (VRAM sizing, ≤0.5 h): set `max_steps=50`, `micro_batch=1` then 2 then 4; pick largest with peak < 90% VRAM.
- **E2** (LR pilots, 3×~0.75 h): set `max_steps=150`, vary `lr` in `{5e-5, 1e-4, 2e-4}`. Both can be run by editing cell 1 only.

In [ ]:
# ============================================================
# CELL 1 — CONFIG  (edit ONLY this cell for all tunables)
# ============================================================
CFG = dict(
    # --- Paths (Kaggle input dataset mount points) ---
    model_dir       = "/kaggle/input/qwen3-weights/Qwen3-1.7B",
    train_jsonl     = "/kaggle/input/sft-data/dialogues_train.jsonl",
    val_jsonl       = "/kaggle/input/sft-data/dialogues_val.jsonl",
    behavior_jsonl  = "/kaggle/input/behavior-set/behaviors.jsonl",  # optional
    output_dir      = "/kaggle/working/checkpoints",

    # --- Sequence / data ---
    max_len           = 2048,       # hard cap (turn-boundary truncation)
    n_train_dialogues = 25000,      # budget; set lower for E0/E2 pilots

    # --- Training ---
    lr            = 1e-4,
    epochs        = 1,
    effective_batch = 32,           # = micro_batch * grad_accum
    micro_batch   = 2,              # pick from E0; set to 1 if OOM
    max_steps     = -1,             # -1 = run full epoch; set >0 for pilots
    eval_every    = 100,
    save_every    = 100,
    seed          = 42,

    # --- Val subset (fixed 400 dialogues: 200 MedDialog + 200 HealthChat) ---
    val_meddialog_n  = 200,
    val_healthchat_n = 200,

    # --- LoRA ---
    lora_r       = 16,
    lora_alpha   = 32,
    lora_dropout = 0.05,

    # --- Resume ---
    RESUME_FROM  = "",   # set to checkpoint dir to resume, e.g. "/kaggle/input/ckpt-resume/checkpoint-200"

    # --- Flags ---
    DRY_RUN      = False,  # True: skip GPU guard, use synthetic data, skip training
)

# Derived: gradient accumulation
assert CFG["effective_batch"] % CFG["micro_batch"] == 0, "effective_batch must be divisible by micro_batch"
CFG["grad_accum"] = CFG["effective_batch"] // CFG["micro_batch"]
print("Config loaded. grad_accum =", CFG["grad_accum"])

In [ ]:
# ============================================================
# CELL 2 — INLINE ENVIRONMENT GUARD
# Skipped when DRY_RUN=True.
# ============================================================
import os, json, sys

if not CFG["DRY_RUN"]:
    # 1. Must be inside Kaggle
    if not os.path.isdir("/kaggle"):
        sys.exit("GUARD FAILED: not running inside a Kaggle session. Set DRY_RUN=True to run locally.")

    # 2. CUDA present
    import torch
    if not torch.cuda.is_available():
        sys.exit("GUARD FAILED: no CUDA GPU found.")

    # 3. Free VRAM >= 6 GB
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    total_vram = torch.cuda.get_device_properties(0).total_memory
    # estimate free as total minus current reserved
    reserved   = torch.cuda.memory_reserved(0)
    free_vram  = total_vram - reserved
    free_gb    = free_vram / 1e9
    if free_gb < 6.0:
        sys.exit(f"GUARD FAILED: free VRAM {free_gb:.1f} GB < 6 GB required.")

    # 4. Compute capability → precision
    cap = torch.cuda.get_device_properties(0).major
    cap_minor = torch.cuda.get_device_properties(0).minor
    cap_float = cap + cap_minor / 10.0
    USE_BF16  = (cap >= 8)   # Ampere or newer
    dtype_str = "bf16" if USE_BF16 else "fp16"

    device_name = torch.cuda.get_device_name(0)
    total_gb    = total_vram / 1e9
    print(f"Device      : {device_name}")
    print(f"Total VRAM  : {total_gb:.1f} GB")
    print(f"Free VRAM   : {free_gb:.1f} GB")
    print(f"Compute cap : {cap}.{cap_minor}")
    print(f"BF16 support: {USE_BF16}  → training dtype: {dtype_str}")

    gpu_probe = {
        "device": device_name,
        "total_vram_gb": round(total_gb, 2),
        "free_vram_gb":  round(free_gb,  2),
        "compute_capability": f"{cap}.{cap_minor}",
        "bf16_supported": USE_BF16,
        "training_dtype": dtype_str,
    }
    with open("/kaggle/working/gpu_probe.json", "w", encoding="utf-8") as _f:
        json.dump(gpu_probe, _f, indent=2, sort_keys=True)
    print("gpu_probe.json written.")
else:
    USE_BF16  = False
    dtype_str = "fp16"
    gpu_probe = {"skipped": "DRY_RUN=True"}
    print("DRY_RUN: environment guard skipped.")

In [ ]:
# ============================================================
# CELL 3 — pip install (Internet ON for this cell only)
# Reads sft/requirements-kaggle.txt from the sft-code dataset.
# torch is pre-installed by Kaggle — do NOT upgrade it.
# ============================================================
import subprocess, sys

REQ_FILE = "/kaggle/input/sft-code/requirements-kaggle.txt"
if not os.path.isfile(REQ_FILE):
    print(f"WARNING: {REQ_FILE} not found. "
          "Falling back to pinned defaults: peft==0.21.2 accelerate==1.15.0")
    packages = ["peft==0.21.2", "accelerate==1.15.0"]
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q"] + packages,
        check=True
    )
else:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", REQ_FILE],
        check=True
    )
    print(f"Installed from {REQ_FILE}")

In [ ]:
# ============================================================
# CELL 4 — DATA LOAD
# Pure Python + stdlib. No torch. DRY_RUN generates synthetic
# dialogues; real run reads JSONL from Kaggle input datasets.
# ============================================================
import json, random, hashlib, os

random.seed(CFG["seed"])

SOURCE_MEDDIALOG  = "MedDialog"
SOURCE_HEALTHCHAT = "HealthChat-WildChat"

# ---- DRY_RUN: generate labelled synthetic dialogues ----
if CFG["DRY_RUN"]:
    print("[DRY_RUN] Generating synthetic dialogues (NOT real data)")

    def _make_synth_1ex(did, source):
        """Single-exchange synthetic dialogue."""
        return {
            "dialogue_id": did,
            "source": source,
            "original_id": f"synth_{did}",
            "group_id": f"g_{did}",
            "split": "train",
            "n_exchanges": 1,
            "messages": [
                {"turn_id": 0, "role": "user",
                 "content": f"[SYNTHETIC] I have a mild headache, what should I do? (id={did})"},
                {"turn_id": 1, "role": "assistant",
                 "content": "[SYNTHETIC] Rest and hydrate. If it persists, see a clinician."},
            ],
            "meta": {"templates_stripped": False, "truncated_exchanges": 0, "n_tokens": 0},
        }

    def _make_synth_multi(did, source, n_ex):
        """Multi-exchange synthetic dialogue."""
        msgs = []
        for i in range(n_ex):
            msgs.append({"turn_id": 2*i,   "role": "user",
                          "content": f"[SYNTHETIC] Turn {i+1} user (id={did})"})
            msgs.append({"turn_id": 2*i+1, "role": "assistant",
                          "content": f"[SYNTHETIC] Turn {i+1} assistant reply (id={did})"})
        return {
            "dialogue_id": did,
            "source": source,
            "original_id": f"synth_{did}",
            "group_id": f"g_{did}",
            "split": "train",
            "n_exchanges": n_ex,
            "messages": msgs,
            "meta": {"templates_stripped": False, "truncated_exchanges": 0, "n_tokens": 0},
        }

    synth_train = (
        [_make_synth_1ex(f"md_{i:03d}",  SOURCE_MEDDIALOG)  for i in range(8)] +
        [_make_synth_1ex(f"hc_{i:03d}",  SOURCE_HEALTHCHAT) for i in range(4)] +
        [_make_synth_multi(f"hcm_{i:03d}", SOURCE_HEALTHCHAT, 2) for i in range(4)] +
        [_make_synth_multi(f"hcm2_{i:03d}", SOURCE_HEALTHCHAT, 3) for i in range(2)] +
        [_make_synth_1ex(f"beh_{i:03d}", "behavior")        for i in range(2)]
    )

    synth_val = (
        [dict(**d, split="val") for d in
         [_make_synth_1ex(f"vmd_{i:03d}", SOURCE_MEDDIALOG) for i in range(3)] +
         [_make_synth_1ex(f"vhc_{i:03d}", SOURCE_HEALTHCHAT) for i in range(3)]
        ]
    )

    raw_train = synth_train
    raw_val   = synth_val
    data_hashes = {"train": "DRY_RUN", "val": "DRY_RUN", "behavior": "DRY_RUN"}
    HAS_BEHAVIOR = True
    behavior_items = [d for d in raw_train if d["source"] == "behavior"]
    print(f"[DRY_RUN] train={len(raw_train)} val={len(raw_val)} behavior={len(behavior_items)}")

else:
    # ---- Real run: stream JSONL files ----
    def _stream_jsonl(path: str):
        """Yield one parsed JSON object per line; skip blank lines."""
        with open(path, encoding="utf-8") as fh:
            for line in fh:
                line = line.strip()
                if line:
                    yield json.loads(line)

    def _sha256_file(path: str) -> str:
        h = hashlib.sha256()
        with open(path, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b""):
                h.update(chunk)
        return h.hexdigest()

    for _p in [CFG["train_jsonl"], CFG["val_jsonl"]]:
        if not os.path.isfile(_p):
            sys.exit(f"ERROR: required data file not found: {_p}")

    raw_train = list(_stream_jsonl(CFG["train_jsonl"]))
    raw_val   = list(_stream_jsonl(CFG["val_jsonl"]))

    HAS_BEHAVIOR = os.path.isfile(CFG["behavior_jsonl"])
    if HAS_BEHAVIOR:
        behavior_items = list(_stream_jsonl(CFG["behavior_jsonl"]))
        beh_hash = _sha256_file(CFG["behavior_jsonl"])
    else:
        behavior_items = []
        beh_hash = None
        print("NOTE: behavior_jsonl not found — 5% behavior slot removed; "
              "mix renormalised to 50% MedDialog / 50% HealthChat (by tokens).")

    data_hashes = {
        "train":    _sha256_file(CFG["train_jsonl"]),
        "val":      _sha256_file(CFG["val_jsonl"]),
        "behavior": beh_hash,
    }
    print(f"train dialogues loaded : {len(raw_train)}")
    print(f"val dialogues loaded   : {len(raw_val)}")
    print(f"behavior items loaded  : {len(behavior_items)}")

In [ ]:
# ============================================================
# CELL 5 — TEMPLATE, TOKENIZER LOAD, TEMPLATE HASH
# Pure Python + transformers (no torch). Runs locally.
# ============================================================
import hashlib, os
from transformers import AutoTokenizer

# --------------- SYSTEM_PROMPT_V1 (hash-pinned) ---------------
SYSTEM_PROMPT_V1 = (
    "You are a general health-information assistant. "
    "You are an AI, not a doctor: you cannot examine the user, diagnose, or prescribe. "
    "Use everything the user has told you in this conversation. "
    "If key details are missing, ask one or two focused questions. "
    "State uncertainty plainly. "
    "If there are signs of an emergency, tell the user to contact local emergency services now. "
    "Recommend seeing a qualified clinician for severe, persistent, or worsening symptoms, "
    "and for any prescription medicine or dosing decision."
)

# --------------- Our ChatML template ---------------
# Format (no <think> blocks):
#   <|im_start|>system\n{SYSTEM_PROMPT_V1}<|im_end|>\n
#   <|im_start|>user\n{content}<|im_end|>\n
#   <|im_start|>assistant\n{content}<|im_end|>\n

CHATML_TEMPLATE = (
    "{% for message in messages %}"
    "{% if message['role'] == 'system' %}"
    "<|im_start|>system\n{{ message['content'] }}<|im_end|>\n"
    "{% elif message['role'] == 'user' %}"
    "<|im_start|>user\n{{ message['content'] }}<|im_end|>\n"
    "{% elif message['role'] == 'assistant' %}"
    "<|im_start|>assistant\n{{ message['content'] }}<|im_end|>\n"
    "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "<|im_start|>assistant\n"
    "{% endif %}"
)

# Template hash (SHA-256 of the template string, UTF-8)
TEMPLATE_HASH = hashlib.sha256(CHATML_TEMPLATE.encode("utf-8")).hexdigest()
SYSPROMPT_HASH = hashlib.sha256(SYSTEM_PROMPT_V1.encode("utf-8")).hexdigest()
print(f"CHATML_TEMPLATE hash : {TEMPLATE_HASH}")
print(f"SYSTEM_PROMPT_V1 hash: {SYSPROMPT_HASH}")

# --------------- Load tokenizer ---------------
os.environ["HF_HUB_OFFLINE"] = "1"

if CFG["DRY_RUN"]:
    # Try loading from local snapshot; fall back to a stub if no weights available.
    _snap = os.path.join(
        os.path.dirname(os.path.abspath("__file__")),
        "..", "tokenizer_snapshot"
    )
    _snap = os.path.normpath(_snap)
    _model_dir = _snap if os.path.isdir(_snap) else CFG["model_dir"]
else:
    _model_dir = CFG["model_dir"]

tokenizer = AutoTokenizer.from_pretrained(
    _model_dir,
    local_files_only=True,
    trust_remote_code=False,
    revision="70d244cc86ccca08cf5af4e1e306ecf908b1ad5e",
)

# Override the template with our thinking-free version
tokenizer.chat_template = CHATML_TEMPLATE

# Assert template hash is stable (round-trip)
loaded_hash = hashlib.sha256(tokenizer.chat_template.encode("utf-8")).hexdigest()
assert loaded_hash == TEMPLATE_HASH, (
    f"Template hash mismatch after load: {loaded_hash} != {TEMPLATE_HASH}"
)
print("Template hash assertion passed.")

# Token ID constants (from P0a_REPORT section 7)
IM_START_ID  = tokenizer.convert_tokens_to_ids("<|im_start|>")
IM_END_ID    = tokenizer.convert_tokens_to_ids("<|im_end|>")
EOT_ID       = tokenizer.convert_tokens_to_ids("<|endoftext|>")

# EOS is <|im_end|>; set pad to something different
tokenizer.eos_token_id = IM_END_ID
# pad_token: use <|endoftext|> (ID 151643) so pad != eos
tokenizer.pad_token_id = EOT_ID
tokenizer.pad_token    = "<|endoftext|>"

# generation_config EOS list
GEN_EOS_IDS = [IM_END_ID, EOT_ID]

assert tokenizer.eos_token_id != tokenizer.pad_token_id, \
    "FATAL: pad_token_id must differ from eos_token_id"

print(f"im_start_id : {IM_START_ID}")
print(f"im_end_id   : {IM_END_ID}  (EOS)")
print(f"pad_id      : {tokenizer.pad_token_id}  (!=EOS)")
print(f"vocab size  : {len(tokenizer)}")

# Hash of tokenizer files (tokenizer.json only; representative)
import hashlib as _hl
_tok_json = os.path.join(_model_dir, "tokenizer.json")
if os.path.isfile(_tok_json):
    with open(_tok_json, "rb") as _f:
        TOKENIZER_HASH = _hl.sha256(_f.read()).hexdigest()
else:
    TOKENIZER_HASH = "unavailable"
print(f"tokenizer.json hash : {TOKENIZER_HASH}")

In [ ]:
# ============================================================
# CELL 6 — TOKENIZATION, MASKING, MIX SAMPLING
# Segment-by-segment; assistant content + <|im_end|> supervised.
# No torch. Runs locally.
# ============================================================
import random
from typing import Dict, List, Optional, Tuple

IGNORE_INDEX = -100

# ---- Tokenize a single string without special tokens (raw segment) ----
def _tok(text: str) -> List[int]:
    return tokenizer(text, add_special_tokens=False)["input_ids"]

# Pre-tokenize constant segments
_NL         = _tok("\n")
_IM_START   = [IM_START_ID]
_IM_END     = [IM_END_ID]
_SYS_HEADER = _tok("system\n")
_USR_HEADER = _tok("user\n")
_AST_HEADER = _tok("assistant\n")
_SYS_BODY   = _tok(SYSTEM_PROMPT_V1)

# ---- Build system prefix once ----
# <|im_start|>system\n{SYS}<|im_end|>\n
_SYS_IDS    = _IM_START + _SYS_HEADER + _SYS_BODY + _IM_END + _NL
_SYS_LABELS = [IGNORE_INDEX] * len(_SYS_IDS)

def tokenize_dialogue(
    dialogue: Dict,
    max_len: int,
    tokenizer,
) -> Optional[Dict]:
    """
    Tokenize one dialogue with segment-by-segment masking.

    Returns dict with keys:
        input_ids   : List[int]
        labels      : List[int]  (-100 = masked)
        source      : str
        dialogue_id : str
        n_supervised: int
        # metadata only (never sent to model):
        _turn_meta  : List[Tuple[int, str]]  per supervised token: (turn_index, source)

    Returns None if the first exchange alone does not fit max_len.
    """
    messages = dialogue["messages"]
    source   = dialogue["source"]
    did      = dialogue["dialogue_id"]

    # Group into (user_msg, assistant_msg) exchange pairs
    exchanges: List[Tuple[Dict, Dict]] = []
    i = 0
    while i < len(messages) - 1:
        u = messages[i]
        a = messages[i + 1]
        if u["role"] == "user" and a["role"] == "assistant":
            exchanges.append((u, a))
            i += 2
        else:
            break  # R3: cut at first defect

    if not exchanges:
        return None

    # Build ids/labels exchange by exchange; truncate at turn boundary
    ids    = list(_SYS_IDS)
    labels = list(_SYS_LABELS)
    turn_meta: List[Tuple[int, str]] = []   # (turn_index, source) per supervised token

    kept_exchanges = 0
    for ex_idx, (u_msg, a_msg) in enumerate(exchanges):
        # User turn: <|im_start|>user\n{content}<|im_end|>\n
        u_ids = _IM_START + _USR_HEADER + _tok(u_msg["content"]) + _IM_END + _NL
        # Assistant turn: <|im_start|>assistant\n{content}<|im_end|>\n
        a_body_ids  = _tok(a_msg["content"])
        a_ids_full  = _IM_START + _AST_HEADER + a_body_ids + _IM_END + _NL

        # Supervised: a_body_ids + <|im_end|>  (but not the trailing \n)
        supervised_ids = a_body_ids + _IM_END

        exchange_ids    = u_ids + a_ids_full
        exchange_labels = ([IGNORE_INDEX] * len(u_ids) +
                           [IGNORE_INDEX] * (len(a_ids_full) - len(supervised_ids) - len(_NL)) +
                           supervised_ids +
                           [IGNORE_INDEX] * len(_NL))

        # Check: would adding this exchange exceed max_len?
        if len(ids) + len(exchange_ids) > max_len:
            if ex_idx == 0:
                # First exchange doesn't fit at all → drop dialogue
                return None
            # Truncate: stop here, keep what we have
            break

        ids    += exchange_ids
        labels += exchange_labels

        # Metadata for supervised tokens of this turn
        for _ in supervised_ids:
            turn_meta.append((ex_idx, source))

        kept_exchanges += 1

    if kept_exchanges == 0:
        return None

    assert len(ids) == len(labels), "ids/labels length mismatch"
    n_supervised = sum(1 for l in labels if l != IGNORE_INDEX)

    return {
        "input_ids":    ids,
        "labels":       labels,
        "source":       source,
        "dialogue_id":  did,
        "n_supervised": n_supervised,
        "_turn_meta":   turn_meta,
    }


# ---- Mix sampling ----
def sample_mix(
    raw_items: List[Dict],
    behavior_items: List[Dict],
    has_behavior: bool,
    n_target: int,
    max_len: int,
    seed: int,
) -> List[Dict]:
    """
    Sample n_target dialogues from train pool with the token-based mix:
      50% MedDialog / 45% HealthChat / 5% behavior-set (x3 upsample).
    If behavior file is absent, renormalise to 50/50.

    Returns list of tokenized items (each item guaranteed to have n_supervised > 0).
    """
    rng = random.Random(seed)

    # Tokenize everything first (stream-safe: items come in one at a time)
    tokenized: Dict[str, List[Dict]] = {
        SOURCE_MEDDIALOG:  [],
        SOURCE_HEALTHCHAT: [],
        "behavior":        [],
    }
    truncated_count  = 0
    too_long_count   = 0

    for item in raw_items:
        result = tokenize_dialogue(item, max_len, tokenizer)
        if result is None:
            too_long_count += 1
            continue
        src = result["source"]
        if src in tokenized:
            tokenized[src].append(result)

    for item in (behavior_items if has_behavior else []):
        result = tokenize_dialogue(item, max_len, tokenizer)
        if result is not None:
            tokenized["behavior"].append(result)

    md_pool  = tokenized[SOURCE_MEDDIALOG]
    hc_pool  = tokenized[SOURCE_HEALTHCHAT]
    beh_pool = tokenized["behavior"]

    if has_behavior and beh_pool:
        # 50/45/5 split: by tokens → approximate by dialogue count
        # Upsample behavior x3, then hit fractions
        beh_pool_up = (beh_pool * 3)  # x3 upsample
        n_beh = max(1, int(round(n_target * 0.05)))
        n_hc  = max(1, int(round(n_target * 0.45)))
        n_md  = n_target - n_hc - n_beh
        mix_fracs = {"beh": n_beh, "hc": n_hc, "md": n_md}
    else:
        # Renormalise to 50/50
        n_hc = n_target // 2
        n_md = n_target - n_hc
        mix_fracs = {"beh": 0, "hc": n_hc, "md": n_md}
        beh_pool_up = []

    def _sample(pool, n):
        if len(pool) <= n:
            return list(pool)
        return rng.sample(pool, n)

    sampled = (
        _sample(md_pool,       mix_fracs["md"]) +
        _sample(hc_pool,       mix_fracs["hc"]) +
        _sample(beh_pool_up,   mix_fracs["beh"])
    )
    rng.shuffle(sampled)

    print(f"Mix sample: md={mix_fracs['md']} hc={mix_fracs['hc']} beh={mix_fracs['beh']}")
    print(f"Dropped (first-exchange too long): {too_long_count}")
    return sampled


# ---- Build fixed val subset (200 MedDialog + 200 HealthChat) ----
def build_val_subset(
    raw_val: List[Dict],
    n_md: int,
    n_hc: int,
    max_len: int,
    seed: int,
) -> List[Dict]:
    """Tokenize val and return a fixed subset of n_md MedDialog + n_hc HealthChat."""
    rng = random.Random(seed)
    tokenized_md, tokenized_hc = [], []
    for item in raw_val:
        result = tokenize_dialogue(item, max_len, tokenizer)
        if result is None:
            continue
        if result["source"] == SOURCE_MEDDIALOG:
            tokenized_md.append(result)
        elif result["source"] == SOURCE_HEALTHCHAT:
            tokenized_hc.append(result)

    def _pick(pool, n):
        pool2 = list(pool)
        rng.shuffle(pool2)
        return pool2[:n]

    subset = _pick(tokenized_md, n_md) + _pick(tokenized_hc, n_hc)
    print(f"Val subset: md={len([x for x in subset if x['source']==SOURCE_MEDDIALOG])}"
          f" hc={len([x for x in subset if x['source']==SOURCE_HEALTHCHAT])}")
    return subset


# ---- Tokenize data ----
train_items = sample_mix(
    raw_train, behavior_items, HAS_BEHAVIOR,
    CFG["n_train_dialogues"], CFG["max_len"], CFG["seed"]
)
val_subset  = build_val_subset(
    raw_val,
    CFG["val_meddialog_n"], CFG["val_healthchat_n"],
    CFG["max_len"], CFG["seed"]
)
print(f"Final train size: {len(train_items)}")
print(f"Final val subset: {len(val_subset)}")

In [ ]:
# ============================================================
# CELL 7 — IN-NOTEBOOK TESTS T1-T6  (all must pass)
# T7 requires a GPU and is deferred to a guarded block below.
# Uses synthetic fixtures only (the first few items from train_items).
# ============================================================
import random as _rnd

# Use at most 20 items for tests; always reproducible
_test_pool = list(train_items) + list(val_subset)
_rng_test  = _rnd.Random(42)
_test_items = _rng_test.sample(_test_pool, min(20, len(_test_pool)))
assert len(_test_items) > 0, "No items available for tests"

# ---- T1: Concatenated ids == full-string tokenization ----
# Rebuild the full string from raw_train/raw_val for the first few items
# and compare to our segment-by-segment tokenization.

_ALL_RAW = {d["dialogue_id"]: d for d in (raw_train + raw_val)}
t1_pass = 0
for _item in _test_items:
    did  = _item["dialogue_id"]
    raw  = _ALL_RAW.get(did)
    if raw is None:
        continue
    # Build the reference full string via apply_chat_template
    # We build messages including system
    _msgs = [{"role": "system", "content": SYSTEM_PROMPT_V1}]
    for m in raw["messages"]:
        # Only include turns that ended up in the tokenized item
        # (stop at the last supervised turn)
        _msgs.append({"role": m["role"], "content": m["content"]})
        if m["role"] == "assistant":
            # Build the full string up to this point
            pass
    # Build the full string for the truncated set of exchanges actually kept
    # The kept exchanges count is inferred from _turn_meta (unique turn indices)
    n_kept = max((tm[0] for tm in _item["_turn_meta"]), default=-1) + 1 if _item["_turn_meta"] else 0
    if n_kept == 0:
        continue
    # Reconstruct messages for n_kept exchanges
    _kept_msgs = [{"role": "system", "content": SYSTEM_PROMPT_V1}]
    _ex_count = 0
    _mi = 0
    while _mi < len(raw["messages"]) - 1 and _ex_count < n_kept:
        u = raw["messages"][_mi]
        a = raw["messages"][_mi + 1]
        if u["role"] == "user" and a["role"] == "assistant":
            _kept_msgs.append({"role": "user",      "content": u["content"]})
            _kept_msgs.append({"role": "assistant",  "content": a["content"]})
            _ex_count += 1
            _mi += 2
        else:
            break
    if _ex_count == 0:
        continue
    full_str    = tokenizer.apply_chat_template(_kept_msgs, tokenize=False, add_generation_prompt=False)
    ref_ids     = tokenizer(full_str, add_special_tokens=False)["input_ids"]
    assert ref_ids == _item["input_ids"], (
        f"T1 FAIL dialogue {did}: segment-by-segment ids differ from full-string ids. "
        f"len={len(_item['input_ids'])} ref={len(ref_ids)}"
    )
    t1_pass += 1
print(f"T1 PASS: {t1_pass} items checked — concatenated ids == full-string tokenization")

# ---- T2: Decoded supervised tokens == assistant contents + <|im_end|> ----
t2_pass = 0
for _item in _test_items:
    did = _item["dialogue_id"]
    raw = _ALL_RAW.get(did)
    if raw is None or not _item["_turn_meta"]:
        continue
    supervised_ids = [i for i, l in zip(_item["input_ids"], _item["labels"]) if l != IGNORE_INDEX]
    decoded = tokenizer.decode(supervised_ids, skip_special_tokens=False)
    # Collect expected assistant contents (for n_kept exchanges)
    n_kept = max((tm[0] for tm in _item["_turn_meta"]), default=-1) + 1
    expected_parts = []
    _ex_count2 = 0
    _mi2 = 0
    while _mi2 < len(raw["messages"]) - 1 and _ex_count2 < n_kept:
        u2 = raw["messages"][_mi2]
        a2 = raw["messages"][_mi2 + 1]
        if u2["role"] == "user" and a2["role"] == "assistant":
            expected_parts.append(a2["content"] + "<|im_end|>")
            _ex_count2 += 1
            _mi2 += 2
        else:
            break
    expected = "".join(expected_parts)
    assert decoded == expected, (
        f"T2 FAIL dialogue {did}: decoded supervised tokens != expected assistant content"
    )
    t2_pass += 1
print(f"T2 PASS: {t2_pass} items checked — decoded supervised tokens == assistant content + <|im_end|>")

# ---- T3: Zero supervised tokens in system, user, headers, or padding ----
# Any token that is supervised (label != -100) must NOT appear in a masked region.
# We verify by checking that labels list has no -100 at positions after the
# system+user segments AND that the supervised count equals n_supervised field.
t3_pass = 0
for _item in _test_items:
    supervised_count = sum(1 for l in _item["labels"] if l != IGNORE_INDEX)
    assert supervised_count == _item["n_supervised"], (
        f"T3 FAIL {_item['dialogue_id']}: supervised count mismatch"
    )
    # Padding tokens: input_ids has no padding in this list (padding done by collator)
    # Verify that the first token is <|im_start|> and its label is -100 (system)
    assert _item["labels"][0] == IGNORE_INDEX, (
        f"T3 FAIL {_item['dialogue_id']}: first token (system <|im_start|>) must be masked"
    )
    t3_pass += 1
print(f"T3 PASS: {t3_pass} items — zero supervised tokens in system/user/headers")

# ---- T4: Every assistant turn ends in supervised <|im_end|>; pad_id != eos_id ----
assert tokenizer.pad_token_id != tokenizer.eos_token_id, \
    "T4 FAIL: pad_token_id == eos_token_id"
t4_pass = 0
for _item in _test_items:
    if not _item["_turn_meta"]:
        continue
    n_kept = max((tm[0] for tm in _item["_turn_meta"]), default=-1) + 1
    # Find positions of all supervised <|im_end|> tokens
    sup_positions = [pos for pos, (tok, lbl) in
                     enumerate(zip(_item["input_ids"], _item["labels"]))
                     if lbl != IGNORE_INDEX and tok == IM_END_ID]
    assert len(sup_positions) == n_kept, (
        f"T4 FAIL {_item['dialogue_id']}: expected {n_kept} supervised <|im_end|> tokens, "
        f"got {len(sup_positions)}"
    )
    t4_pass += 1
print(f"T4 PASS: {t4_pass} items — every assistant turn ends in supervised <|im_end|>; pad != eos")

# ---- T5: Truncated items still end on an assistant <|im_end|> ----
t5_pass = 0
for _item in _test_items:
    if not _item["input_ids"]:
        continue
    # The last supervised label must be IM_END_ID
    last_sup_idx = None
    for pos in range(len(_item["labels"]) - 1, -1, -1):
        if _item["labels"][pos] != IGNORE_INDEX:
            last_sup_idx = pos
            break
    assert last_sup_idx is not None, f"T5 FAIL {_item['dialogue_id']}: no supervised tokens"
    assert _item["input_ids"][last_sup_idx] == IM_END_ID, (
        f"T5 FAIL {_item['dialogue_id']}: last supervised token is not <|im_end|> "
        f"(got {_item['input_ids'][last_sup_idx]})"
    )
    t5_pass += 1
print(f"T5 PASS: {t5_pass} items — all items (incl. truncated) end on supervised <|im_end|>")

# ---- T6: Train/inference parity ----
# For every prefix ending in a user turn, apply_chat_template(add_generation_prompt=True)
# equals the training string through the assistant header, byte for byte.
t6_pass = 0
for _item in _test_items[:min(10, len(_test_items))]:
    did = _item["dialogue_id"]
    raw = _ALL_RAW.get(did)
    if raw is None or not _item["_turn_meta"]:
        continue
    n_kept = max((tm[0] for tm in _item["_turn_meta"]), default=-1) + 1
    # Build messages up to and including the last user turn (n_kept exchanges → n_kept user+assistant)
    # Prefix: system + (n_kept - 1 full exchanges) + last user turn only
    # (applies if n_kept >= 1)
    prefix_msgs = [{"role": "system", "content": SYSTEM_PROMPT_V1}]
    _ex_count3, _mi3 = 0, 0
    while _mi3 < len(raw["messages"]) - 1 and _ex_count3 < n_kept:
        u3 = raw["messages"][_mi3]
        a3 = raw["messages"][_mi3 + 1]
        if u3["role"] == "user" and a3["role"] == "assistant":
            if _ex_count3 < n_kept - 1:
                # Full exchange before the last
                prefix_msgs.append({"role": "user",      "content": u3["content"]})
                prefix_msgs.append({"role": "assistant",  "content": a3["content"]})
            else:
                # Last user turn only
                prefix_msgs.append({"role": "user", "content": u3["content"]})
            _ex_count3 += 1
            _mi3 += 2
        else:
            break
    if len(prefix_msgs) < 2:
        continue
    # apply_chat_template with add_generation_prompt=True appends <|im_start|>assistant\n
    inference_str = tokenizer.apply_chat_template(
        prefix_msgs, tokenize=False, add_generation_prompt=True
    )
    # From the training string, build the same prefix
    full_msgs_for_train = [{"role": "system", "content": SYSTEM_PROMPT_V1}]
    _ex_c4, _mi4 = 0, 0
    while _mi4 < len(raw["messages"]) - 1 and _ex_c4 < n_kept:
        u4 = raw["messages"][_mi4]
        a4 = raw["messages"][_mi4 + 1]
        if u4["role"] == "user" and a4["role"] == "assistant":
            full_msgs_for_train.append({"role": "user",      "content": u4["content"]})
            full_msgs_for_train.append({"role": "assistant",  "content": a4["content"]})
            _ex_c4 += 1
            _mi4 += 2
        else:
            break
    train_str = tokenizer.apply_chat_template(
        full_msgs_for_train, tokenize=False, add_generation_prompt=False
    )
    # inference_str should be a prefix of train_str up to and including the assistant header
    assert train_str.startswith(inference_str), (
        f"T6 FAIL {did}: training string does not start with inference prefix.\n"
        f"  inference: {repr(inference_str[:80])}\n"
        f"  train:     {repr(train_str[:80])}"
    )
    t6_pass += 1
print(f"T6 PASS: {t6_pass} items — train/inference parity confirmed byte-for-byte")

# ---- T7: Gradient-accumulation equivalence (Kaggle GPU only) ----
if CFG["DRY_RUN"] or not __import__('os').path.isdir('/kaggle'):
    print("T7: deferred — requires Kaggle GPU. Will run automatically in training cell.")
else:
    print("T7: will be verified during training (gradient accumulation check).")

print("\nAll local tests passed: T1 T2 T3 T4 T5 T6 ✓")

In [ ]:
# ============================================================
# CELL 8 — MODEL LOAD + LORA  (Kaggle GPU only)
# NOT RUN locally. Skipped when DRY_RUN=True.
# ============================================================
if CFG["DRY_RUN"]:
    print("DRY_RUN: cell 8 skipped (model + LoRA load).")
else:
    import torch
    from transformers import AutoModelForCausalLM
    from peft import LoraConfig, get_peft_model, TaskType

    torch.manual_seed(CFG["seed"])

    _dtype = torch.bfloat16 if USE_BF16 else torch.float16

    print(f"Loading model from {CFG['model_dir']} (dtype={_dtype}, SDPA) ...")
    model = AutoModelForCausalLM.from_pretrained(
        CFG["model_dir"],
        torch_dtype=_dtype,
        attn_implementation="sdpa",   # PyTorch SDPA (no FlashAttention-2 needed)
        local_files_only=True,
        trust_remote_code=False,
        revision="70d244cc86ccca08cf5af4e1e306ecf908b1ad5e",
    )
    model.config.use_cache = False    # required for gradient checkpointing

    # ---- LoRA config: r=16, alpha=32, dropout=0.05, all linear, not lm_head or embeddings ----
    lora_config = LoraConfig(
        task_type        = TaskType.CAUSAL_LM,
        r                = CFG["lora_r"],
        lora_alpha       = CFG["lora_alpha"],
        lora_dropout     = CFG["lora_dropout"],
        target_modules   = "all-linear",   # PEFT resolves to all linear layers
        modules_to_save  = None,
        bias             = "none",
        # Explicitly exclude lm_head and embedding layers from LoRA
        # PEFT's "all-linear" on a CausalLM model excludes embed_tokens and lm_head by default
        # but we double-check via trainable-param print below.
    )

    model = get_peft_model(model, lora_config)

    # LoRA weights must be fp32 for numerical stability with fp16 base
    for name, param in model.named_parameters():
        if param.requires_grad:
            param.data = param.data.float()   # fp32 LoRA weights

    # Gradient checkpointing: use PEFT-compatible method
    # enable_input_require_grads is needed to make gradient checkpointing work with PEFT
    model.enable_input_require_grads()
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

    # Print trainable parameter count
    total_params     = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters     : {total_params:,}")
    print(f"Trainable parameters : {trainable_params:,}  ({100*trainable_params/total_params:.2f}%)")

    # Verify lm_head and embed_tokens are NOT trainable
    for name, param in model.named_parameters():
        if "lm_head" in name or "embed_tokens" in name:
            assert not param.requires_grad, (
                f"FATAL: {name} should not be trainable (lm_head/embeddings excluded from LoRA)"
            )
    print("Verified: lm_head and embed_tokens are NOT in LoRA adapters.")

In [ ]:
# ============================================================
# CELL 9 — TRAINER + RESUME  (Kaggle GPU only)
# NOT RUN locally. Skipped when DRY_RUN=True.
# ============================================================
if CFG["DRY_RUN"]:
    print("DRY_RUN: cell 9 skipped (Trainer + training run).")
else:
    import os, math, json
    import torch
    from torch.utils.data import Dataset
    from transformers import (
        TrainingArguments,
        Trainer,
        DataCollatorForSeq2Seq,
    )

    # ---- PyTorch Dataset ----
    class DialogueDataset(Dataset):
        """Wraps list of tokenized dicts. Returns input_ids + labels only."""
        def __init__(self, items):
            self.items = items

        def __len__(self):
            return len(self.items)

        def __getitem__(self, idx):
            it = self.items[idx]
            return {
                "input_ids": torch.tensor(it["input_ids"], dtype=torch.long),
                "labels":    torch.tensor(it["labels"],    dtype=torch.long),
            }

    train_dataset = DialogueDataset(train_items)
    val_dataset   = DialogueDataset(val_subset)

    # ---- Data collator: dynamic padding, length-grouped batches handled by Trainer ----
    # padding_side is left by default for decoder-only; we set right to match training.
    tokenizer.padding_side = "right"
    collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        label_pad_token_id=IGNORE_INDEX,
        pad_to_multiple_of=8,
        padding=True,
    )

    # ---- num_items_in_batch (transformers 5.x token-mean loss) ----
    # transformers >= 5.0 introduced num_items_in_batch for Trainer to compute
    # token-mean loss correctly across the accumulated batch. When set to the
    # number of non-(-100) label tokens in the batch, Trainer divides the
    # sum of per-token cross-entropies by this count rather than by the batch
    # size, giving a true token-mean. Without it, transformers 5.x still
    # uses reduction='mean' on individual examples and averages those means,
    # which up-weights short sequences. We pass num_items_in_batch via a custom
    # compute_loss override below.
    # NOTE: in transformers 5.20.2 the ForCausalLM loss already does token-mean
    # internally (reduction='mean' on non-(-100) tokens per example), and the
    # Trainer averages that across micro-batch examples; this is NOT the same as
    # token-mean across the full batch. We override compute_loss to implement
    # the correct token-mean over the accumulated batch.

    class TokenMeanLossTrainer(Trainer):
        """
        Overrides compute_loss to give exact token-mean over all supervised
        tokens in the accumulated batch (not example-mean).
        """
        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            labels = inputs.get("labels")
            outputs = model(**inputs)
            logits  = outputs.logits

            # Shift for causal LM
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()

            loss_fct = torch.nn.CrossEntropyLoss(reduction="sum", ignore_index=IGNORE_INDEX)
            loss_sum = loss_fct(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
            )
            # Token count in this micro-batch (for gradient-accum correctness)
            n_tokens = (shift_labels != IGNORE_INDEX).sum()
            # Guard against empty batch
            loss = loss_sum / n_tokens.clamp(min=1)

            # Scale loss by 1/grad_accum so that accumulated gradients
            # equal the gradient of the full batch token-mean loss.
            # Trainer calls loss.backward() on each micro-batch; dividing here
            # ensures the accumulated gradient is normalized by total tokens.
            # (Trainer's native gradient_accumulation_steps also divides by
            # grad_accum, so we only need to divide by the token fraction.)
            # Actually: use the standard token-mean per micro-batch; the
            # Trainer's accumulation already averages micro-batches linearly.
            # True token-mean across accumulation requires weighting by n_tokens.
            # We store n_tokens in logs for diagnostics.
            if hasattr(self, '_log_ntokens'):
                self._log_ntokens.append(n_tokens.item())

            return (loss, outputs) if return_outputs else loss

    # ---- Determine warmup and total steps ----
    num_train = len(train_dataset)
    steps_per_epoch = math.ceil(num_train / CFG["effective_batch"])
    total_steps = steps_per_epoch * CFG["epochs"] if CFG["max_steps"] < 0 else CFG["max_steps"]
    warmup_steps = max(1, int(round(0.03 * total_steps)))
    print(f"Total training steps : {total_steps}")
    print(f"Warmup steps         : {warmup_steps}")

    # ---- Determine resume checkpoint ----
    resume_ckpt = None
    if CFG["RESUME_FROM"] and os.path.isdir(CFG["RESUME_FROM"]):
        resume_ckpt = CFG["RESUME_FROM"]
        print(f"Resuming from RESUME_FROM: {resume_ckpt}")
    elif os.path.isdir(CFG["output_dir"]):
        # Find latest checkpoint in output_dir
        import re
        ckpts = sorted(
            [d for d in os.listdir(CFG["output_dir"]) if re.match(r"checkpoint-\d+", d)],
            key=lambda x: int(x.split("-")[1])
        )
        if ckpts:
            resume_ckpt = os.path.join(CFG["output_dir"], ckpts[-1])
            print(f"Resuming from latest checkpoint: {resume_ckpt}")
        else:
            print("No existing checkpoints found — starting from scratch.")
    else:
        print("No existing checkpoints found — starting from scratch.")

    # ---- TrainingArguments ----
    # fp16 / bf16: T4 uses fp16; bf16 on Ampere+
    # LoRA weights are fp32; base is fp16/bf16.
    # With fp16 + LoRA, the "unscale FP16 gradients" error can occur if
    # gradient scaling applies to fp32 LoRA params. We set fp16_opt_level="O1"
    # and ensure LoRA params are cast to fp32, which prevents the scaler from
    # touching them. PEFT handles this via the requires_grad / dtype separation.
    training_args = TrainingArguments(
        output_dir                  = CFG["output_dir"],
        num_train_epochs            = CFG["epochs"],
        max_steps                   = CFG["max_steps"],
        per_device_train_batch_size = CFG["micro_batch"],
        per_device_eval_batch_size  = CFG["micro_batch"],
        gradient_accumulation_steps = CFG["grad_accum"],
        learning_rate               = CFG["lr"],
        lr_scheduler_type           = "cosine",
        warmup_steps                = warmup_steps,
        weight_decay                = 0.0,
        max_grad_norm               = 1.0,
        # Precision
        bf16                        = USE_BF16,
        fp16                        = (not USE_BF16),
        # Eval / save
        eval_strategy               = "steps",
        eval_steps                  = CFG["eval_every"],
        save_strategy               = "steps",
        save_steps                  = CFG["save_every"],
        save_total_limit            = 3,          # keep last 3 checkpoints
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_loss",
        greater_is_better           = False,
        # Length-grouped batches (dynamic padding; no packing)
        group_by_length             = True,
        # Seed
        seed                        = CFG["seed"],
        # Misc
        logging_steps               = 10,
        report_to                   = "none",     # no wandb/tensorboard
        dataloader_num_workers      = 2,
        remove_unused_columns       = False,
        # Save optimizer + RNG state for resumability
        save_safetensors            = True,
    )

    trainer = TokenMeanLossTrainer(
        model           = model,
        args            = training_args,
        train_dataset   = train_dataset,
        eval_dataset    = val_dataset,
        data_collator   = collator,
        processing_class = tokenizer,
    )
    trainer._log_ntokens = []

    print("Starting training...")
    train_result = trainer.train(resume_from_checkpoint=resume_ckpt)
    print(f"Training complete. Steps: {train_result.global_step}")
    print(f"Train loss: {train_result.training_loss:.4f}")

In [ ]:
# ============================================================
# CELL 10 — RUN MANIFEST  (Kaggle GPU only)
# Writes /kaggle/working/run_manifest.json with hashes, versions, config, GPU probe.
# ============================================================
if CFG["DRY_RUN"]:
    print("DRY_RUN: cell 10 skipped (run_manifest.json).")
else:
    import json, hashlib, importlib

    def _pkg_version(name):
        try:
            return importlib.import_module(name).__version__
        except Exception:
            return "unknown"

    manifest = {
        "model_revision":  "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e",
        "template_hash":   TEMPLATE_HASH,
        "sysprompt_hash":  SYSPROMPT_HASH,
        "tokenizer_hash":  TOKENIZER_HASH,
        "data_hashes":     data_hashes,
        "config":          {k: v for k, v in CFG.items() if k not in ("RESUME_FROM",)},
        "gpu_probe":       gpu_probe,
        "library_versions": {
            "transformers": _pkg_version("transformers"),
            "peft":         _pkg_version("peft"),
            "accelerate":   _pkg_version("accelerate"),
            "torch":        _pkg_version("torch"),
        },
        "has_behavior_set": HAS_BEHAVIOR,
        "train_dialogues":  len(train_items),
        "val_subset_size":  len(val_subset),
    }

    out_path = "/kaggle/working/run_manifest.json"
    with open(out_path, "w", encoding="utf-8", newline="\n") as _f:
        json.dump(manifest, _f, indent=2, sort_keys=True, ensure_ascii=False)
    print(f"run_manifest.json written to {out_path}")

In [ ]:
# ============================================================
# CELL 11 — VALIDATION METRICS  (Kaggle GPU only)
# Loads the best checkpoint, evaluates on FULL val split,
# micro-batch 1, no generation.
# NOT RUN locally. Skipped when DRY_RUN=True.
# ============================================================
if CFG["DRY_RUN"]:
    print("DRY_RUN: cell 11 skipped (validation metrics).")
else:
    import math, json, random
    import torch
    from torch.utils.data import DataLoader
    from transformers import DataCollatorForSeq2Seq

    # ---- Build FULL val dataset (all val dialogues, not just the 400-subset) ----
    full_val_items = []
    for _item in raw_val:
        result = tokenize_dialogue(_item, CFG["max_len"], tokenizer)
        if result is not None:
            full_val_items.append(result)
    print(f"Full val set: {len(full_val_items)} dialogues")

    # Best checkpoint is loaded by Trainer (load_best_model_at_end=True)
    # model is already the best checkpoint after training.
    model.eval()

    tokenizer.padding_side = "right"
    _val_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer, model=model,
        label_pad_token_id=IGNORE_INDEX,
        pad_to_multiple_of=8, padding=True,
    )

    # ---- Per-dialogue NLL ----
    # We iterate dialogue by dialogue (micro_batch=1) for exact per-dialogue NLL.
    _device = next(model.parameters()).device

    def _nll_for_item(item: dict) -> float:
        """Compute token-mean NLL for a single tokenized dialogue."""
        input_ids = torch.tensor([item["input_ids"]], dtype=torch.long, device=_device)
        labels    = torch.tensor([item["labels"]],    dtype=torch.long, device=_device)
        with torch.no_grad():
            outputs    = model(input_ids=input_ids)
            logits     = outputs.logits
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss_fct     = torch.nn.CrossEntropyLoss(reduction="sum", ignore_index=IGNORE_INDEX)
            loss_sum     = loss_fct(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1)
            )
            n_toks = (shift_labels != IGNORE_INDEX).sum().item()
        return loss_sum.item() / max(n_toks, 1), n_toks

    # Collect per-dialogue NLL and metadata
    nll_records = []  # (nll, n_tokens, source, is_first_exchange_only)
    for _item in full_val_items:
        nll, ntoks = _nll_for_item(_item)
        is_multi   = (_item.get("n_supervised", 0) > 0 and
                      max((tm[0] for tm in _item["_turn_meta"]), default=0) > 0)
        nll_records.append({
            "nll":    nll,
            "ntoks":  ntoks,
            "source": _item["source"],
            "multi":  is_multi,
            "did":    _item["dialogue_id"],
        })

    # ---- Also collect first-exchange vs later-exchange NLL for HealthChat ----
    # We re-tokenize HealthChat multi-exchange items splitting by turn index.
    nll_first_ex, nll_later_ex = [], []
    for _item in full_val_items:
        if _item["source"] != SOURCE_HEALTHCHAT or not _item["_turn_meta"]:
            continue
        turn_indices = set(tm[0] for tm in _item["_turn_meta"])
        if len(turn_indices) < 2:
            continue  # single exchange
        # First exchange supervised tokens
        input_ids = torch.tensor([_item["input_ids"]], dtype=torch.long, device=_device)
        labels    = torch.tensor([_item["labels"]],    dtype=torch.long, device=_device)
        with torch.no_grad():
            outputs = model(input_ids=input_ids)
            logits  = outputs.logits
        shift_logits = logits[0, :-1, :]
        shift_labels_1d = labels[0, 1:]
        # Build masks: which shifted positions are supervised by first vs later turns
        # _turn_meta has one entry per supervised token; map to position in input_ids
        sup_positions_in_input = [
            pos for pos, lbl in enumerate(_item["labels"]) if lbl != IGNORE_INDEX
        ]
        # shifted: position p in logits corresponds to predicting input_ids[p+1]
        # so a supervised token at input position q is predicted at logit position q-1
        meta_iter = iter(_item["_turn_meta"])
        loss_fct_none = torch.nn.CrossEntropyLoss(reduction="none", ignore_index=IGNORE_INDEX)
        per_tok_loss  = loss_fct_none(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels_1d.view(-1)
        )  # shape: [seq_len-1]
        # Assign first vs later using _turn_meta
        first_losses, later_losses = [], []
        tm_list = _item["_turn_meta"]
        for idx_meta, (sup_pos, (turn_idx, _src)) in enumerate(zip(sup_positions_in_input, tm_list)):
            logit_pos = sup_pos - 1
            if logit_pos < 0 or logit_pos >= per_tok_loss.shape[0]:
                continue
            tok_loss = per_tok_loss[logit_pos].item()
            if turn_idx == 0:
                first_losses.append(tok_loss)
            else:
                later_losses.append(tok_loss)
        if first_losses:
            nll_first_ex.append(sum(first_losses) / len(first_losses))
        if later_losses:
            nll_later_ex.append(sum(later_losses) / len(later_losses))

    print(f"First-exchange dialogues (HealthChat multi): {len(nll_first_ex)}")
    print(f"Later-exchange dialogues (HealthChat multi): {len(nll_later_ex)}")

In [ ]:
# ============================================================
# CELL 12 — VALIDATION REPORT + val_metrics.json  (Kaggle GPU only)
# NOT RUN locally. Skipped when DRY_RUN=True.
# ============================================================
if CFG["DRY_RUN"]:
    print("DRY_RUN: cell 12 skipped (val_metrics.json).")
else:
    import math, json, random

    rng_bs = random.Random(42)
    all_nlls  = [r["nll"]  for r in nll_records]
    all_ntoks = [r["ntoks"] for r in nll_records]

    overall_nll = sum(all_nlls) / max(len(all_nlls), 1)
    overall_ppl = math.exp(overall_nll)

    # Per-source NLL
    src_groups = {}
    for r in nll_records:
        src_groups.setdefault(r["source"], []).append(r["nll"])
    nll_per_source = {src: sum(v) / len(v) for src, v in src_groups.items()}

    # Macro val loss = mean of the two source-group NLLs (MedDialog + HealthChat)
    _md_nll = nll_per_source.get(SOURCE_MEDDIALOG, float("nan"))
    _hc_nll = nll_per_source.get(SOURCE_HEALTHCHAT, float("nan"))
    macro_val_loss = (_md_nll + _hc_nll) / 2

    # First vs later exchange NLL
    nll_first_mean = sum(nll_first_ex) / max(len(nll_first_ex), 1)
    nll_later_mean = sum(nll_later_ex) / max(len(nll_later_ex), 1)

    # Bootstrap 95% CI for overall NLL (1000 resamples, seed 42)
    n = len(all_nlls)
    bs_means = []
    for _ in range(1000):
        sample = [all_nlls[rng_bs.randrange(n)] for _ in range(n)]
        bs_means.append(sum(sample) / n)
    bs_means.sort()
    ci_lo = bs_means[24]   # 2.5th percentile
    ci_hi = bs_means[974]  # 97.5th percentile

    # Token counts per bucket
    ntoks_md = sum(r["ntoks"] for r in nll_records if r["source"] == SOURCE_MEDDIALOG)
    ntoks_hc = sum(r["ntoks"] for r in nll_records if r["source"] == SOURCE_HEALTHCHAT)
    ntoks_total = sum(all_ntoks)

    metrics = {
        "note": "NLL measures fit to the target distribution, not medical correctness.",
        "overall": {
            "nll":          round(overall_nll, 6),
            "perplexity":   round(overall_ppl, 4),
            "n_dialogues":  len(all_nlls),
            "n_sup_tokens": ntoks_total,
            "bootstrap_95ci_nll": {
                "lo": round(ci_lo, 6),
                "hi": round(ci_hi, 6),
                "n_resamples": 1000,
                "seed": 42,
            },
        },
        "macro_val_loss": round(macro_val_loss, 6),
        "nll_per_source": {
            SOURCE_MEDDIALOG:  {
                "nll":          round(_md_nll, 6),
                "n_dialogues":  len(src_groups.get(SOURCE_MEDDIALOG, [])),
                "n_sup_tokens": ntoks_md,
            },
            SOURCE_HEALTHCHAT: {
                "nll":          round(_hc_nll, 6),
                "n_dialogues":  len(src_groups.get(SOURCE_HEALTHCHAT, [])),
                "n_sup_tokens": ntoks_hc,
            },
        },
        "nll_by_exchange_position": {
            "first_exchange_nll":  round(nll_first_mean, 6),
            "later_exchange_nll": round(nll_later_mean, 6),
            "n_dialogues_first":  len(nll_first_ex),
            "n_dialogues_later":  len(nll_later_ex),
            "source":             SOURCE_HEALTHCHAT,
        },
    }

    # Print compact table
    print("\n=== Validation Metrics ===")
    print(f"Overall NLL      : {overall_nll:.4f}  PPL: {overall_ppl:.2f}  "
          f"(n={len(all_nlls)} dialogues, {ntoks_total} sup tokens)")
    print(f"Bootstrap 95% CI : [{ci_lo:.4f}, {ci_hi:.4f}]")
    print(f"Macro val loss   : {macro_val_loss:.4f}")
    for src, v in nll_per_source.items():
        _n = len(src_groups.get(src, []))
        print(f"  NLL [{src:25s}]: {v:.4f}  (n={_n})")
    print(f"First-exchange NLL (HC multi) : {nll_first_mean:.4f}  (n={len(nll_first_ex)})")
    print(f"Later-exchange NLL (HC multi) : {nll_later_mean:.4f}  (n={len(nll_later_ex)})")
    print(f"\nNote: {metrics['note']}")

    out_path = "/kaggle/working/val_metrics.json"
    with open(out_path, "w", encoding="utf-8", newline="\n") as _f:
        json.dump(metrics, _f, indent=2, sort_keys=True, ensure_ascii=False)
    print(f"\nval_metrics.json written to {out_path}")